## Caso del laboratorio

# 1. Carga y exploración básica del corpus

Primero se carga el archivo `spa_shp.csv`. El corpus ya se encuentra normalizado, por lo que solo se realizará una preparación básica: eliminar filas vacías, convertir los textos a string y remover espacios innecesarios.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, GRU, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

# Carga directa del corpus paralelo.
# El archivo debe llamarse exactamente spa_shp.csv y contener las columnas spa y shp.

df = pd.read_csv('spa_shp.csv')

# Preparación básica.
df = df[['spa', 'shp']].copy()
df = df.dropna(subset=['spa', 'shp'])

df['spa'] = df['spa'].astype(str).str.strip().str.lower()
df['shp'] = df['shp'].astype(str).str.strip().str.lower()

# Eliminamos filas vacías si existieran.
df = df[(df['spa'] != '') & (df['shp'] != '')].copy()

df.head()

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

# Estadísticas descriptivas básicas.

num_pares = len(df)
num_duplicados = df.duplicated(subset=['spa', 'shp']).sum()

spa_lengths = df['spa'].str.len()
shp_lengths = df['shp'].str.len()

print('Número de pares:', num_pares)
print('Pares duplicados:', num_duplicados)
print()
print('Longitud promedio en español:', round(spa_lengths.mean(), 2))
print('Longitud mínima en español:', spa_lengths.min())
print('Longitud máxima en español:', spa_lengths.max())
print()
print('Longitud promedio en shipibo:', round(shp_lengths.mean(), 2))
print('Longitud mínima en shipibo:', shp_lengths.min())
print('Longitud máxima en shipibo:', shp_lengths.max())

In [ ]:
# Revisamos algunos pares del corpus.

df.sample(10, random_state=SEED)

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df['spa'].str.len(), bins=30, alpha=0.7, label='Español')
plt.hist(df['shp'].str.len(), bins=30, alpha=0.7, label='Shipibo')
plt.title('Distribución de longitudes por caracteres')
plt.xlabel('Número de caracteres')
plt.ylabel('Frecuencia')
plt.legend()
plt.show()

# 2. Preparación de datos y tokenización

El modelo trabajará a nivel de caracteres. Esta decisión es útil en escenarios de bajo recurso porque reduce el tamaño del vocabulario y permite que el modelo aprenda patrones subléxicos.

Para mantener el laboratorio dentro del tiempo disponible, se usará una cantidad limitada de pares del corpus.

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

# Configuración principal del experimento.

MAX_PAIRS = 4000
EMBEDDING_DIM = 128
LATENT_DIM = 192
BATCH_SIZE = 64
EPOCHS = 20

# Seleccionamos una muestra del corpus para que el entrenamiento sea manejable.
df_model = df.sample(n=min(MAX_PAIRS, len(df)), random_state=SEED).reset_index(drop=True)

print('Pares usados para el entrenamiento:', len(df_model))

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

# En el decoder agregamos tokens especiales:
# \t indica inicio de secuencia.
# \n indica fin de secuencia.

X = df_model['spa'].tolist()
y = ['\t' + text + '\n' for text in df_model['shp'].tolist()]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED
)

print('Train:', len(X_train))
print('Validación:', len(X_val))
print('Test:', len(X_test))

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

# Tokenización a nivel de caracteres.

spa_tokenizer = Tokenizer(char_level=True, filters='', lower=False)
shp_tokenizer = Tokenizer(char_level=True, filters='', lower=False)

spa_tokenizer.fit_on_texts(X_train)
shp_tokenizer.fit_on_texts(y_train)

num_encoder_tokens = len(spa_tokenizer.word_index) + 1
num_decoder_tokens = len(shp_tokenizer.word_index) + 1

MAX_LEN_SPA = max(len(text) for text in X_train)
MAX_LEN_SHP = max(len(text) for text in y_train)

print('Caracteres únicos en español:', num_encoder_tokens)
print('Caracteres únicos en shipibo:', num_decoder_tokens)
print('Longitud máxima español:', MAX_LEN_SPA)
print('Longitud máxima shipibo:', MAX_LEN_SHP)

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

def encode_and_pad(tokenizer, texts, max_len):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

encoder_input_train = encode_and_pad(spa_tokenizer, X_train, MAX_LEN_SPA)
encoder_input_val = encode_and_pad(spa_tokenizer, X_val, MAX_LEN_SPA)
encoder_input_test = encode_and_pad(spa_tokenizer, X_test, MAX_LEN_SPA)

decoder_input_train = encode_and_pad(shp_tokenizer, y_train, MAX_LEN_SHP)
decoder_input_val = encode_and_pad(shp_tokenizer, y_val, MAX_LEN_SHP)
decoder_input_test = encode_and_pad(shp_tokenizer, y_test, MAX_LEN_SHP)

# El target del decoder es la misma secuencia, pero desplazada un carácter hacia la izquierda.
decoder_target_train = np.zeros_like(decoder_input_train)
decoder_target_val = np.zeros_like(decoder_input_val)
decoder_target_test = np.zeros_like(decoder_input_test)

decoder_target_train[:, :-1] = decoder_input_train[:, 1:]
decoder_target_val[:, :-1] = decoder_input_val[:, 1:]
decoder_target_test[:, :-1] = decoder_input_test[:, 1:]

print('encoder_input_train:', encoder_input_train.shape)
print('decoder_input_train:', decoder_input_train.shape)
print('decoder_target_train:', decoder_target_train.shape)

# 3. Modelo Seq2Seq y entrenamiento guiado

Se entrenará un modelo encoder-decoder pequeño con GRU.

- El **encoder** recibe la oración en español.
- El **decoder** aprende a generar la traducción en shipibo-konibo carácter por carácter.
- El estado final del encoder se usa como estado inicial del decoder.

La estructura general del modelo ya está dada. Ustedes deben completar los argumentos principales de las capas y la compilación del modelo.

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

# En esta celda se construye un modelo Seq2Seq sencillo.
# El encoder lee la oración en español y resume la información en un estado oculto.
# El decoder usa ese estado para generar la traducción en shipibo-konibo carácter por carácter.

# Encoder
encoder_inputs = Input(shape=(None,), name='encoder_inputs')

# Pista:
# input_dim debe ser el tamaño del vocabulario de español.
# output_dim debe ser la dimensión de los embeddings definida previamente.
encoder_embedding = Embedding(
    input_dim=num_encoder_tokens,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name='encoder_embedding'
)(encoder_inputs)

# Pista:
# La cantidad de unidades de la GRU.
encoder_gru = GRU(
    LATENT_DIM,
    return_state=True,
    name='encoder_gru'
)
_, encoder_state = encoder_gru(encoder_embedding)

# Decoder
decoder_inputs = Input(shape=(None,), name='decoder_inputs')

# Pista:
# input_dim debe ser el tamaño del vocabulario de shipibo-konibo.
# output_dim debe ser la dimensión de los embeddings definida previamente.
decoder_embedding_layer = Embedding(
    input_dim=num_decoder_tokens,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name='decoder_embedding'
)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

# Pista:
# El decoder también usa LATENT_DIM unidades.
# return_sequences=True permite obtener una predicción para cada posición de la secuencia.
# return_state=True permite recuperar el estado final del decoder.
decoder_gru = GRU(
    LATENT_DIM,
    return_sequences=True,
    return_state=True,
    name='decoder_gru'
)
decoder_outputs, _ = decoder_gru(decoder_embedding, initial_state=encoder_state)

# Capa final:
# Debe producir una distribución de probabilidad sobre el vocabulario shipibo-konibo.
# Por eso, el número de neuronas debe ser el tamaño del vocabulario de salida.
# La activación debe ser softmax.
decoder_dense = Dense(num_decoder_tokens, activation='softmax', name='decoder_dense')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Pista:
# Usen Adam como optimizador.
# Como el target está codificado como enteros, usen sparse_categorical_crossentropy.
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

# EarlyStopping detiene el entrenamiento si la pérdida de validación deja de mejorar en 4 iteraciones
# Completen solo los valores principales.
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    [encoder_input_train, decoder_input_train],
    decoder_target_train,
    validation_data=(
        [encoder_input_val, decoder_input_val],
        decoder_target_val
    ),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.title('Evolución de la pérdida')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='Train accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.title('Evolución de la exactitud por carácter')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# 4. Traducción, comparación y evaluación automática

En esta sección se usan funciones ya definidas para generar traducciones y calcular métricas automáticas. No es necesario implementar desde cero greedy decoding, beam search ni BLEU.

El objetivo es que prueben el modelo, comparen salidas y expliquen qué muestran las métricas.

In [ ]:
# Modelos de inferencia.

encoder_model = Model(encoder_inputs, encoder_state)

decoder_state_input = Input(shape=(LATENT_DIM,), name='decoder_state_input')
decoder_single_input = Input(shape=(1,), name='decoder_single_input')

decoder_single_embedding = decoder_embedding_layer(decoder_single_input)
decoder_single_outputs, decoder_single_state = decoder_gru(
    decoder_single_embedding,
    initial_state=decoder_state_input
)
decoder_single_outputs = decoder_dense(decoder_single_outputs)

decoder_model = Model(
    [decoder_single_input, decoder_state_input],
    [decoder_single_outputs, decoder_single_state]
)

reverse_shp_index = {idx: char for char, idx in shp_tokenizer.word_index.items()}
start_token_id = shp_tokenizer.word_index['\t']
end_token_id = shp_tokenizer.word_index['\n']

In [ ]:
def translate_greedy(sentence):
    """Traduce una oración usando greedy decoding."""
    input_seq = encode_and_pad(spa_tokenizer, [sentence], MAX_LEN_SPA)
    state = encoder_model.predict(input_seq, verbose=0)

    current_token = np.array([[start_token_id]])
    generated = []

    for _ in range(MAX_LEN_SHP):
        output_tokens, state = decoder_model.predict([current_token, state], verbose=0)
        sampled_token_id = int(np.argmax(output_tokens[0, -1, :]))

        if sampled_token_id == end_token_id or sampled_token_id == 0:
            break

        generated.append(reverse_shp_index.get(sampled_token_id, ''))
        current_token = np.array([[sampled_token_id]])

    return ''.join(generated).strip()

In [ ]:
def translate_beam(sentence, beam_width=2):
    """Traduce una oración usando beam search con un ancho pequeño."""
    input_seq = encode_and_pad(spa_tokenizer, [sentence], MAX_LEN_SPA)
    initial_state = encoder_model.predict(input_seq, verbose=0)

    # Cada candidato contiene: secuencia de IDs, puntaje acumulado y estado del decoder.
    sequences = [([start_token_id], 0.0, initial_state)]

    for _ in range(MAX_LEN_SHP):
        all_candidates = []

        for seq, score, state in sequences:
            last_token = seq[-1]

            if last_token == end_token_id:
                all_candidates.append((seq, score, state))
                continue

            current_token = np.array([[last_token]])
            output_tokens, new_state = decoder_model.predict([current_token, state], verbose=0)
            probs = output_tokens[0, -1, :]

            # Tomamos solo los mejores candidatos para reducir el tiempo de evaluación.
            top_ids = np.argsort(probs)[-beam_width:]

            for token_id in top_ids:
                token_prob = float(probs[token_id])
                candidate_score = score - np.log(token_prob + 1e-9)
                all_candidates.append((seq + [int(token_id)], candidate_score, new_state))

        sequences = sorted(all_candidates, key=lambda x: x[1])[:beam_width]

        if all(seq[-1] == end_token_id for seq, _, _ in sequences):
            break

    best_seq = sequences[0][0]
    chars = []

    for token_id in best_seq:
        if token_id in [start_token_id, end_token_id, 0]:
            continue
        chars.append(reverse_shp_index.get(token_id, ''))

    return ''.join(chars).strip()

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

# Probamos algunas traducciones del conjunto de prueba.
# Para cada ejemplo, recuperen:
# - src: oración en español desde X_test.
# - ref: traducción de referencia desde y_test, quitando \t y \n.
# - hyp: traducción generada por translate_greedy(src).

N_EXAMPLES = 5

for i in range(min(N_EXAMPLES, len(X_test))):
    src = X_test[i]
    ref = y_test[i].replace('\t', '').replace('\n', '').strip()
    hyp = translate_beam(src)

    print('ES  :', src)
    print('REF :', ref)
    print('HYP :', hyp)
    print('-' * 60)

In [ ]:
# COMPLETAR EL CÓDIGO AQUÍ

# Comparación breve entre greedy decoding y beam search.
# Greedy elige siempre el carácter más probable.
# Beam search mantiene varias opciones candidatas; aquí usamos beam_width pequeño para ahorrar tiempo.

N_BEAM_EXAMPLES = 3
BEAM_WIDTH = 2

for i in range(min(N_BEAM_EXAMPLES, len(X_test))):
    src = X_test[i]
    ref = y_test[i].replace('\t', '').replace('\n', '').strip()

    hyp_greedy = translate_greedy(src)
    hyp_beam = translate_beam(src, beam_width=BEAM_WIDTH)

    print('ES      :', src)
    print('REF     :', ref)
    print('GREEDY  :', hyp_greedy)
    print('BEAM    :', hyp_beam)
    print('-' * 60)

In [ ]:
def exact_match_score(references, hypotheses):
    matches = [ref.strip() == hyp.strip() for ref, hyp in zip(references, hypotheses)]
    return np.mean(matches)


def get_char_ngrams(text, n):
    return [text[i:i+n] for i in range(len(text) - n + 1)]


def char_bleu_score(reference, hypothesis, max_n=4):
    """Calcula una versión sencilla de BLEU a nivel de caracteres."""
    reference = reference.strip()
    hypothesis = hypothesis.strip()

    if len(hypothesis) == 0:
        return 0.0

    precisions = []

    for n in range(1, max_n + 1):
        ref_ngrams = get_char_ngrams(reference, n)
        hyp_ngrams = get_char_ngrams(hypothesis, n)

        if len(hyp_ngrams) == 0:
            precisions.append(0.0)
            continue

        ref_counts = {}
        for ng in ref_ngrams:
            ref_counts[ng] = ref_counts.get(ng, 0) + 1

        overlap = 0
        for ng in hyp_ngrams:
            if ref_counts.get(ng, 0) > 0:
                overlap += 1
                ref_counts[ng] -= 1

        # Suavizado simple para evitar puntajes cero cuando un n-grama no coincide.
        precisions.append((overlap + 1) / (len(hyp_ngrams) + 1))

    geo_mean = np.exp(np.mean(np.log(precisions)))

    if len(hypothesis) < len(reference):
        brevity_penalty = np.exp(1 - len(reference) / max(len(hypothesis), 1))
    else:
        brevity_penalty = 1.0

    return float(brevity_penalty * geo_mean)

In [ ]:
# Evaluación del modelo.
# Para mantener el tiempo controlado, beam search usa menos ejemplos.

USE_BEAM_FOR_EVAL = True
BEAM_WIDTH_EVAL = 2

if USE_BEAM_FOR_EVAL:
    N_EVAL = min(10, len(X_test))
else:
    N_EVAL = min(20, len(X_test))

references = []
hypotheses = []

for i in range(N_EVAL):
    src = X_test[i]
    ref = y_test[i].replace('\t', '').replace('\n', '').strip()

    if USE_BEAM_FOR_EVAL:
        hyp = translate_beam(src, beam_width=BEAM_WIDTH_EVAL)
    else:
        hyp = translate_greedy(src)

    references.append(ref)
    hypotheses.append(hyp)

em = exact_match_score(references, hypotheses)
bleu_scores = [char_bleu_score(r, h) for r, h in zip(references, hypotheses)]

method_name = f'beam search, beam_width={BEAM_WIDTH_EVAL}' if USE_BEAM_FOR_EVAL else 'greedy'

print(f'Decodificación usada: {method_name}')
print(f'Ejemplos evaluados: {N_EVAL}')
print(f'Exact Match: {em:.4f}')
print(f'BLEU promedio a nivel de caracteres: {np.mean(bleu_scores):.4f}')
print()

for i in range(min(10, N_EVAL)):
    print('ES :', X_test[i])
    print('REF:', references[i])
    print('HYP:', hypotheses[i])
    print('-' * 60)